In [1]:
import json 
from mksense.config import PROCESSED_DATA_DIR
repo = "scikit-learn"
doc_input_path = PROCESSED_DATA_DIR / repo / f"{repo}_docs_cleaned.json"
with open(doc_input_path, 'rt') as  f_in:
    documents = json.load(f_in)

2025-08-24 10:14:29.239 | INFO     | mksense.config:<module>:11 - PROJ_ROOT path is: /workspaces/mksense


In [2]:
documents[0]

{'content': '.. _supervised-learning: Supervised learning ------------------- .. toctree:: :maxdepth: 2 modules/linear_model modules/lda_qda.rst modules/kernel_ridge.rst modules/svm modules/sgd modules/neighbors modules/gaussian_process modules/cross_decomposition.rst modules/naive_bayes modules/tree modules/ensemble modules/multiclass modules/feature_selection.rst modules/semi_supervised.rst modules/isotonic.rst modules/calibration.rst modules/neural_networks_supervised',
 'file_path': 'supervised_learning.rst',
 'title': 'Supervised learning',
 'context': '',
 'extension': 'rst',
 'repo': 'scikit-learn'}

In [3]:
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
base_url=os.environ.get("BASE_URL")
api_key=os.environ.get("API_KEY")

llm_client = OpenAI(
    base_url=base_url,
    api_key=api_key,
)

In [4]:
def build_prompt(query, search_results):
    prompt_template = """
    Your are a vertern software developer. 
    Answer the QUESTION based on the CONTEXT from the documentation database. 
    Use only the facts from the CONTEXT when answering the QUESTION. 
    If the CONTEXT doesn't containt the answer output None.

    QUESTION:
    {question}

    CONTEXT:
    {context}
    """.strip()
    
    context = ""

    for doc in search_results:
        context = context + f"repo: {doc['repo']}\ntitle:{doc['title']}\ndocument: {doc['content']}\n\n"

    prompt = prompt_template.format(question=query, context=context).strip()
    
    return prompt 


In [5]:
def llm(prompt):
    responce = llm_client.chat.completions.create(
        model="deepseek/deepseek-r1-0528:free",
        messages=[{
            "role":"user",
            "content":prompt
        }]
    )
    return responce.choices[0].message.content

In [6]:
query = "how do I install scikit-learn"

```bash
 docker run -it \
   --rm \
   --name elasticsearch \
   -m 4GB \
   -p 9200:9200 \
   -p 9300:9300 \
   -e "discovery.type=single-node" \
   -e "xpack.security.enabled=false" \
   elasticsearch:9.1.0
```

run elastic searrch with this docker code

In [7]:
from elasticsearch import Elasticsearch
es_client = Elasticsearch('http://localhost:9200')

In [8]:
es_client.info()

ObjectApiResponse({'name': 'a1600b4df7b4', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'nWuIBhBzSkWUB1HEjbIixA', 'version': {'number': '9.1.0', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '00e7d33bf08f1476229d9d1642e2da46cfebdd53', 'build_date': '2025-07-23T22:09:53.891289976Z', 'build_snapshot': False, 'lucene_version': '10.2.2', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'})

In [9]:
index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "content": {"type": "text"},
            "file_path": {"type": "text"},
            "title": {"type": "text"},
            "context": {"type": "text"},
            "extension": {"type": "text"},
            "repo": {"type": "text"} 
        }
    }
}

index_name = "sklearn-docs"

es_client.indices.create(index=index_name, body=index_settings)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'sklearn-docs'})

In [10]:
from tqdm.auto import tqdm
for doc in tqdm(documents):
    es_client.index(index=index_name, document=doc)

/opt/conda/envs/mksense/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
 15%|█▌        | 16/106 [00:00<00:01, 51.59it/s]

100%|██████████| 106/106 [00:01<00:00, 64.75it/s]


In [11]:
def elastic_search(query):
    search_query = {
        "size": 5,
        "query": {
            "bool": {
                "must": {
                    "multi_match": {
                        "query": query,
                        "fields": ["title^3", "content", "context"],
                        "type": "best_fields"
                    }
                },
                "filter": {
                    "term": {
                        "repo": "scikit-learn"
                    }
                }
            }
        }
    }

    response = es_client.search(index=index_name, body=search_query)
    
    result_docs = []
    
    for hit in response['hits']['hits']:
        result_docs.append(hit['_source'])
    
    return result_docs

In [12]:
def rag(query):
    search_results = elastic_search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [13]:
query = "how do I install scikit-learn?"
rag(query)

'Based solely on the provided context, which contains **no information** about installing scikit-learn:\n\n**Answer:** None'

```bash
docker pull qdrant/qdrant

docker run -p 6333:6333 -p 6334:6334 \
   -v "$(pwd)/data/qdrant_storage:/qdrant/storage:z" \
   qdrant/qdrant
```

start the qd_client

In [14]:
from qdrant_client import QdrantClient, models
from mksense.config import PROCESSED_DATA_DIR

qd_client = QdrantClient('http://localhost:6333')

EMBEDDING_DIMENSIONALITY  = 512

model_handle = "jinaai/jina-embeddings-v2-small-en"
 
# Define the collection name
collection_name = 'scikit-learn_docs'


In [15]:
qd_client.delete_collection(collection_name=collection_name)

True

In [16]:

# Create the collection with the specific vector parameters
qd_client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=EMBEDDING_DIMENSIONALITY, # Dimensionality of the vectors
        distance=models.Distance.COSINE # Distance metrics of the vectors
    )
)

True

In [17]:
qd_client.create_payload_index(
    collection_name=collection_name,
    field_name="repo",
    field_schema="keyword"
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [18]:
documents[0]

{'content': '.. _supervised-learning: Supervised learning ------------------- .. toctree:: :maxdepth: 2 modules/linear_model modules/lda_qda.rst modules/kernel_ridge.rst modules/svm modules/sgd modules/neighbors modules/gaussian_process modules/cross_decomposition.rst modules/naive_bayes modules/tree modules/ensemble modules/multiclass modules/feature_selection.rst modules/semi_supervised.rst modules/isotonic.rst modules/calibration.rst modules/neural_networks_supervised',
 'file_path': 'supervised_learning.rst',
 'title': 'Supervised learning',
 'context': '',
 'extension': 'rst',
 'repo': 'scikit-learn'}

In [19]:
points = []

for i, doc in enumerate(documents):
    text = doc['title'] + ' ' + doc['content'] 
    vector = models.Document(text=text, model=model_handle)
    point = models.PointStruct(
        id=i,
        vector=vector, 
        payload=doc  
    )
    points.append(point)

In [ ]:
points[0]

PointStruct(id=0, vector=Document(text='Supervised learning .. _supervised-learning: Supervised learning ------------------- .. toctree:: :maxdepth: 2 modules/linear_model modules/lda_qda.rst modules/kernel_ridge.rst modules/svm modules/sgd modules/neighbors modules/gaussian_process modules/cross_decomposition.rst modules/naive_bayes modules/tree modules/ensemble modules/multiclass modules/feature_selection.rst modules/semi_supervised.rst modules/isotonic.rst modules/calibration.rst modules/neural_networks_supervised', model='jinaai/jina-embeddings-v2-small-en', options=None), payload={'content': '.. _supervised-learning: Supervised learning ------------------- .. toctree:: :maxdepth: 2 modules/linear_model modules/lda_qda.rst modules/kernel_ridge.rst modules/svm modules/sgd modules/neighbors modules/gaussian_process modules/cross_decomposition.rst modules/naive_bayes modules/tree modules/ensemble modules/multiclass modules/feature_selection.rst modules/semi_supervised.rst modules/isot

: 

In [ ]:
qd_client.upsert(
    collection_name=collection_name,
    points=points
)

In [ ]:
def vector_search(query):
    print('Vector Seaach is Used')
    course = 'data-engineering-zoomcamp'
    query_points = qd_client.query_points(
        collection_name=collection_name,
        query=models.Document(  #embedded the query text locally with jinaai
            text=query,
            model=model_handle
        ),
        query_filter=models.Filter(
            must=[
                models.FieldCondition(
                    key="course",
                    match=models.MatchValue(value=course)
                )
            ]
        ),
        limit=5,
        with_payload=True
    )

    results = []

    for point in query_points.points:
        results.append(point.payload)
    
    return results 

In [ ]:
search_results = vector_search(query)


In [ ]:
def rag(query):
    search_results = vector_search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [ ]:
query = "how do I run kafka?"
rag(query)